# Data preparation & hierarchical model notebook

Continues directly from the data-loading cell Salma set up (`raw = pd.read_csv(...)`).
Each section below follows the same pattern: **what we tried** -> **what we found** -> **what's decided** (or still open).


In [ ]:
import pandas as pd
from pathlib import Path

REPO_URL = "https://github.com/salmaelhanchi/NeuroMatch_2026_Behaviour.git"
REPO_DIR = Path("/content/NeuroMatch_2026_Behaviour")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

matches = list(REPO_DIR.rglob("data01_direction4priors.csv"))

if not matches:
    raise FileNotFoundError("data01_direction4priors.csv was not found inside the cloned repo.")

CSV_PATH = matches[0]
raw = pd.read_csv(CSV_PATH)

print(f"Loaded from: {CSV_PATH}")
print(f"raw shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head()


## 1. Building the trial-level dataset

`estimate_x`/`estimate_y` encode the subject's response as a coordinate pair, not a plain angle
(same idea as `motion_direction`, just represented differently on screen). `np.arctan2` recovers
the angle correctly because it looks at both signs at once -- plain `arctan(y/x)` can't tell
opposite directions apart, since they give the same ratio.

Circular error can't be plain subtraction either, for the same reason `error_deg` needed the
`% 360` trick: 355 degrees and 5 degrees are 10 degrees apart on the circle, not 350.


In [ ]:
import numpy as np

data = raw.copy()

# --- convert response vector to an angle --------------------------------
data['estimate_deg'] = np.degrees(np.arctan2(data.estimate_y, data.estimate_x)) % 360
data.loc[data.estimate_deg == 0, 'estimate_deg'] = 360

n_before = len(data)
data = data.dropna(subset=['estimate_deg', 'motion_direction']).copy()
print(f"dropped {n_before - len(data)} rows with missing estimate")

# --- circular estimation error -------------------------------------------
# shift by 180 before the modulo, then shift back -- handles both directions
# of wraparound without an if-statement.
data['error_deg'] = ((data.estimate_deg - data.motion_direction + 180) % 360) - 180

# --- block id --------------------------------------------------------------
# run_id ALONE is not a unique block: it resets to 1 for every subject, so
# grouping by run_id alone lumps "subject 1's first block" together with
# "subject 2's first block". (subject_id, run_id) together is the real,
# unique block identifier -- combined here into one string column.
data['block_id'] = data['subject_id'].astype(str) + '_' + data['run_id'].astype(str)

trial_data = data.rename(columns={
    'subject_id': 'subject',
    'trial_index': 'trial',
    'prior_std': 'prior_width',
    'motion_coherence': 'coherence',
    'motion_direction': 'true_direction',
})[['subject', 'block_id', 'trial', 'prior_width', 'coherence',
    'true_direction', 'estimate_deg', 'error_deg',
    'session_id', 'experiment_id']]

print(f"trial_data shape: {trial_data.shape}")
trial_data.head()


**What we tried:** grouping by `run_id` alone to check block sizes -- gave nonsense block
sizes in the thousands, because `run_id` resets per subject.

**What we found:** `(subject_id, run_id)` blocks range 204-226 trials (paper reports 202-226),
and each block has exactly one `prior_std` value. Confirmed below.

**Decided:** block = `(subject_id, run_id)`, stored as a single `block_id` string column.


In [ ]:
block_sizes = data.groupby(['subject_id', 'run_id']).size()
print('block size range:', block_sizes.min(), '-', block_sizes.max())

prior_check = data.groupby(['subject_id', 'run_id'])['prior_std'].nunique()
print('blocks with more than one prior_std value (should be 0):', (prior_check > 1).sum())


## 2. What `experiment_id` actually is

**What we tried:** checked whether `experiment_id == 12` was a separate 5-subject follow-up
study, as originally theorized.

**What we found:** it isn't. `experiment_id` 12 appears for 10 of the 12 subjects (not 5), and
summing sessions across `experiment_id` 11 + 12 for each subject matches the paper's reported
session counts exactly (8, 8, 9, 5, 6, 7, 6, 6, 8, 6, 6, 6 for subjects 1-12). It looks like a
database/registration split of one single study, not two different studies.

**Decided:** treat `experiment_id` 11 and 12 as one combined dataset per subject. The
five-subject-follow-up theory is resolved as incorrect.


In [ ]:
session_check = trial_data.groupby('subject')['session_id'].nunique()
print(session_check)
print('paper reports: 8, 8, 9, 5, 6, 7, 6, 6, 8, 6, 6, 6')


## 3. Data-quality caveat: `trial_index` and `trial_time` are block-relative, not global

**What we tried:** measure the real time gap between the last trial of one block and the first
trial of the next, to check whether blocks run back-to-back within a session (this matters for
the carryover hypothesis -- if there's no break, "bleeding" between blocks is physically
plausible).

**What we found:** both `trial_index` and `trial_time` reset to 1 / 0.0 at the start of every
block. Sorting a subject's rows by raw `trial_index` does **not** give true chronological order,
it groups "everyone's trial #1 of any block" together. This also means the raw file has **no way
to measure the real wall-clock gap between blocks** -- `trial_time` resets right at the boundary,
erasing that information. (Confirmed directly below for subject 2.)

**Decided (for now):** rely on the paper's Methods text (no rest period is described between
blocks within a session) as the best available evidence that blocks run consecutively.

**Still open:** confirm with Salma/Varsha whether a source with real inter-block timestamps
exists anywhere outside this CSV. Any feature that depends on trial order (previous block,
previous trial, RT trends) must sort by `(run_id, trial_index)` within a subject, never raw
`trial_index` alone -- and subjects with data split across two `experiment_id`s (1, 3, 5, 7, 8)
need extra care, since `trial_index` isn't unique across that split either.


In [ ]:
demo = data[data.subject_id == 2].sort_values(['run_id', 'trial_index'])
print(demo[['run_id', 'trial_index', 'trial_time']].iloc[208:214].to_string(index=False))


## 4. Hierarchical model: reliability-controlled switching

Varsha's settled hypothesis (13 Jul meeting): *"The observer first estimates how reliable the
prior itself is, and this estimate (hyper-prior) controls switching."*

**What we tried first (rejected):** a continuous-width hyperprior that multiplies prior x
likelihood together (the model in `hb_verified_model_implementation.ipynb`, using
`prior_kappa_t`). This is unimodal by construction -- a product of two von Mises distributions
is always a single von Mises -- so it cannot reproduce the paper's central bimodality finding.
The team explicitly moved away from this construction.

**What we're building instead:** a discrete mixture. Each trial, the observer reports from the
prior component with probability `prior_reliance`, or from the likelihood component with
probability `1 - prior_reliance`. `prior_reliance` is a hidden state, learned trial-by-trial via
a delta rule (deliberately not named `prior_confidence`, since that name is already used by the
rejected continuous-width version -- reusing it would blur two different mechanisms together).


In [ ]:
# borrowed from switching_bayesian_observer_starter.py -- needed by trial_percept_distribution
from scipy.special import i0e

DEG = np.arange(1, 361)

def vm_pdf(support_deg, mu_deg, k, norm=True):
    mu = np.atleast_1d(np.asarray(mu_deg, float))
    x  = np.deg2rad(np.asarray(support_deg, float))[:, None]
    u  = np.deg2rad(mu)[None, :]
    k  = float(k)
    if np.isinf(k) or k > 1e300:
        out = np.zeros((len(support_deg), len(mu)))
        for j, mm in enumerate(mu):
            out[np.argmin(np.abs(np.asarray(support_deg) - mm)), j] = 1.0
    else:
        out = np.exp(k * np.cos(x - u) - k) / (2 * np.pi * i0e(k))
    if norm:
        out = out / out.sum(0, keepdims=True)
    return out


In [ ]:
def wrap_signed_deg(diff_deg):
    # same formula derived by hand for error_deg
    return ((diff_deg + 180) % 360) - 180


def circular_mean_deg(angles_deg):
    """
    Circular mean of several angles. Can't average degrees directly, same
    wraparound issue as error_deg, so convert to (x, y) unit vectors first,
    average those, then convert back with arctan2.
    """
    angles_rad = np.deg2rad(np.asarray(angles_deg))
    mean_x = np.mean(np.cos(angles_rad))
    mean_y = np.mean(np.sin(angles_rad))
    return float(np.degrees(np.arctan2(mean_y, mean_x)) % 360)


def prior_agreement(measurement_deg, prior_mean_deg, k_prior):
    """
    How consistent was this trial's sensory measurement with what the prior
    expected, on a 0-to-1 scale? 1.0 = measurement landed exactly on the
    prior mean. Decays toward 0 as the measurement moves away.
    """
    delta_rad = np.deg2rad(wrap_signed_deg(measurement_deg - prior_mean_deg))
    return float(np.exp(k_prior * (np.cos(delta_rad) - 1.0)))


def update_prior_reliance(prior_reliance, measurement_deg, prior_mean_deg, k_prior, alpha):
    """
    Delta-rule update for prior_reliance: how much the observer currently
    relies on the prior, learned trial by trial, in [0, 1].

        prior_reliance_next = prior_reliance + alpha * (agreement_t - prior_reliance)
    """
    agreement_t = prior_agreement(measurement_deg, prior_mean_deg, k_prior)
    reliance_next = prior_reliance + alpha * (agreement_t - prior_reliance)
    return float(np.clip(reliance_next, 1e-4, 1.0 - 1e-4)), agreement_t


def trial_percept_distribution(prior_mean_deg, measurement_deg, k_prior, k_llh, prior_reliance):
    """
    P(percept | trial) as a genuine MIXTURE (added, never multiplied), so two
    peaks survive whenever prior_reliance sits away from 0 or 1:

        P(percept) = prior_reliance       * VonMises(percept; prior_mean,  k_prior)
                   + (1 - prior_reliance) * VonMises(percept; measurement, k_llh)
    """
    prior_component = vm_pdf(DEG, prior_mean_deg, k_prior)[:, 0]
    llh_component   = vm_pdf(DEG, measurement_deg, k_llh)[:, 0]
    mixture = prior_reliance * prior_component + (1 - prior_reliance) * llh_component
    return mixture / mixture.sum()


### Sanity check: does `prior_reliance` move the right direction?

Toy sequence: three trials agreeing with the prior, three clashing, three agreeing again.


In [ ]:
prior_mean, k_prior, alpha = 225, 8, 0.2
prior_reliance = 0.5

measurements = [225, 220, 230,   45, 50, 40,   228, 222, 226]
for m in measurements:
    prior_reliance, agreement_t = update_prior_reliance(prior_reliance, m, prior_mean, k_prior, alpha)
    print(f"measurement={m:>3}  agreement={agreement_t:.3f}  prior_reliance={prior_reliance:.3f}")


**Result:** reliance climbed 0.5 -> 0.73 across three agreeing trials, dropped to 0.375
across three clashing trials, climbed back to 0.68 once agreement resumed. Direction of movement
is correct.

### Real block-transition test

Used the paper's own concentration values (STAR Methods): k=33.3 for a 10-degree prior, k=0.7
for an 80-degree prior. Simulated 20 trials of a narrow block followed by 20 trials of a wide
block, drawing directions from `np.random.default_rng(7).vonmises(...)` for realism.

**Found:** with `alpha=0.2`, `prior_reliance` swung noisily (0.44-0.84) even during the
genuinely-predictable narrow block, single noisy trials could yank reliance around, because the
yardstick for "surprising" comes from the same sharp `k_prior` as the block's true concentration.


In [ ]:
rng = np.random.default_rng(7)
k_prior_10, k_prior_80 = 33.3, 0.7
prior_mean_val = 225
mu_rad = np.deg2rad(prior_mean_val)

block_A = (np.rad2deg(rng.vonmises(mu_rad, k_prior_10, 20)) + 360) % 360
block_B = (np.rad2deg(rng.vonmises(mu_rad, k_prior_80, 20)) + 360) % 360

reliance = 0.5
trajectory = []
for m in block_A:
    reliance, _ = update_prior_reliance(reliance, m, prior_mean_val, k_prior_10, alpha=0.2)
    trajectory.append(reliance)
for m in block_B:
    reliance, _ = update_prior_reliance(reliance, m, prior_mean_val, k_prior_80, alpha=0.2)
    trajectory.append(reliance)

print('peak in 10-degree block:', round(max(trajectory[:20]), 3))
print('worst dip in 10-degree block:', round(min(trajectory[:20]), 3))
print('range in 80-degree block:', round(max(trajectory[20:]) - min(trajectory[20:]), 3))


**Tried lowering alpha -- a real tradeoff, not a fix.** Lower alpha calmed the noise (range
narrowed from a 0.4-point swing down to about 0.05 at `alpha=0.05`) but also weakened how high
`prior_reliance` could climb within a typical block, a genuine speed-vs-stability tradeoff, not
resolved by tuning alpha alone.

**Tried smoothing agreement over a short recent-trial window instead -- this worked better.**
Judging agreement against the circular mean of the last 5 measurements (not the newest one
alone) let `prior_reliance` climb fast (still `alpha=0.2`) *and* stay stable: peak 0.95 in the
narrow block (vs. 0.84 before), worst dip 0.58 (vs. 0.44 before).

**Open design question:** in that test the 5-trial window wasn't reset at the block boundary, so
a few trials of memory naturally bled across into the new block -- an unplanned but plausible
proxy for carryover. Needs a deliberate decision: reset the window at each new block, or keep
this bleed-over on purpose as a stand-in for the carryover mechanism.


In [ ]:
from collections import deque

def update_prior_reliance_smoothed(prior_reliance, window_buf, measurement_deg,
                                     prior_mean_deg, k_prior, alpha):
    """
    Same delta rule as update_prior_reliance, but agreement is judged against
    the circular mean of the last few measurements (window_buf), not the
    newest one alone -- a single noisy trial can no longer swing
    prior_reliance by itself.
    """
    window_buf.append(measurement_deg)
    smoothed_measurement = circular_mean_deg(list(window_buf))
    agreement_t = prior_agreement(smoothed_measurement, prior_mean_deg, k_prior)
    reliance_next = prior_reliance + alpha * (agreement_t - prior_reliance)
    return float(np.clip(reliance_next, 1e-4, 1.0 - 1e-4)), agreement_t


reliance = 0.5
window_buf = deque(maxlen=5)
trajectory_smoothed = []
for m, kp in zip(list(block_A) + list(block_B), [k_prior_10] * 20 + [k_prior_80] * 20):
    reliance, _ = update_prior_reliance_smoothed(reliance, window_buf, m, prior_mean_val, kp, alpha=0.2)
    trajectory_smoothed.append(reliance)

print('peak in 10-degree block:', round(max(trajectory_smoothed[:20]), 3))
print('worst dip in 10-degree block:', round(min(trajectory_smoothed[:20]), 3))


### Deciding when the window resets: session boundaries, not every block

Resetting the window at *every* new block would make it impossible to ever detect a carryover
effect, carried-over trials are exactly what a within-session transition should let bleed across
the boundary. Building "never bleeds" into the window's plumbing would quietly answer the
project's own research question before any fitting happens.

The paper's design gives a real, *verified* boundary to reset on, though: sessions happen on
separate days, while blocks within a session run back-to-back with no described break. That maps
directly onto the `same_session_prev` convention already used in
`hb_verified_model_implementation.ipynb`'s `build_block_context`.

**Decided:** reset the window only at session boundaries (`same_session_prev == False`, or a
subject's very first block). Within a session, let the window carry across block boundaries
naturally, that's the mechanism the model is meant to test, not something to prevent by default.


In [ ]:
# --- session-boundary flag, needed to decide when the window resets --------
block_context = (
    data.groupby(['subject_id', 'run_id'], as_index=False)
    .agg(session_id=('session_id', 'first'))
    .sort_values(['subject_id', 'run_id'])
    .reset_index(drop=True)
)
block_context['prev_session_id'] = block_context.groupby('subject_id')['session_id'].shift(1)
block_context['same_session_prev'] = (
    block_context['prev_session_id'].notna()
    & (block_context['session_id'] == block_context['prev_session_id'])
)
block_context['block_id'] = block_context['subject_id'].astype(str) + '_' + block_context['run_id'].astype(str)

# also brings run_id back onto trial_data (needed for correct chronological sorting;
# trial_index alone resets every block, run_id is the real per-subject block-order key)
trial_data = trial_data.merge(
    block_context[['block_id', 'run_id', 'same_session_prev']], on='block_id', how='left'
)

print(trial_data['same_session_prev'].value_counts(dropna=False))


In [ ]:
def update_prior_reliance_windowed(prior_reliance, window_buf, measurement_deg,
                                     prior_mean_deg, k_prior, alpha,
                                     reset_window=False):
    """
    Same mechanism as update_prior_reliance_smoothed, but can be told to clear
    the window first. Used only at session boundaries, where a real, verified
    gap (separate days) means nothing should carry over. Within a session,
    reset_window stays False and the window carries across block boundaries
    naturally, exactly the mechanism being tested.
    """
    if reset_window:
        window_buf.clear()
    window_buf.append(measurement_deg)
    smoothed_measurement = circular_mean_deg(list(window_buf))
    agreement_t = prior_agreement(smoothed_measurement, prior_mean_deg, k_prior)
    reliance_next = prior_reliance + alpha * (agreement_t - prior_reliance)
    return float(np.clip(reliance_next, 1e-4, 1.0 - 1e-4)), agreement_t


### Checking the reset logic on a real subject

Runs subject 1's trials in true chronological order (`run_id`, then `trial`, since `trial_index`
alone resets every block) and logs, at each new block, whether `same_session_prev` fired a reset.


In [ ]:
demo = trial_data[trial_data.subject == 1].sort_values(['run_id', 'trial']).reset_index(drop=True)

reliance = 0.5
window_buf = deque(maxlen=5)
prev_block_id = None
reset_log = []

for _, row in demo.iterrows():
    is_new_block = row.block_id != prev_block_id
    reset_now = is_new_block and not bool(row.same_session_prev)
    reliance, _ = update_prior_reliance_windowed(
        reliance, window_buf, row.estimate_deg, 225, 8, alpha=0.2, reset_window=reset_now
    )
    if is_new_block:
        reset_log.append((row.block_id, bool(row.same_session_prev), reset_now))
    prev_block_id = row.block_id

print(f'{"block_id":>10} {"same_session_prev":>18} {"window reset?":>14}')
for b, s, r in reset_log[:10]:
    print(f'{b:>10} {str(s):>18} {str(r):>14}')


## 5. Varsha's first concrete check: does Kp/Ke behave as predicted?

From the 13 Jul meeting, the team's agreed immediate first step, separate from the reliability-
mixture model above, and much simpler:

"Use the existing switching model to examine Kp Ke, the confidence or weight assigned to the
prior, separately for 6%, 12%, and 24% motion coherence... lower coherence leads to greater
reliance on the prior; higher coherence leads to less reliance on the prior."

**What we tried:** fit the paper's own Switching observer (`girshick_lookup`/`trial_loglike`,
ported from `switching_bayesian_observer_starter.py`) to each subject's real trials, fitting `Ke`
(`k_llh`) separately per coherence and `Kp` (`k_prior`) separately per prior width. Renamed the
resulting ratio `condition_prior_reliance = Kp/(Kp+Ke)` rather than the paper's bare `p_prior`,
since it's the same underlying idea as `prior_reliance` above, just fixed per condition here
instead of learned trial-by-trial.


In [ ]:
from scipy.special import i0e
from scipy.optimize import minimize

DEG = np.arange(1, 361)

def vm_pdf(support_deg, mu_deg, k, norm=True):
    mu = np.atleast_1d(np.asarray(mu_deg, float))
    x  = np.deg2rad(np.asarray(support_deg, float))[:, None]
    u  = np.deg2rad(mu)[None, :]
    k  = float(k)
    if np.isinf(k) or k > 1e300:
        out = np.zeros((len(support_deg), len(mu)))
        for j, mm in enumerate(mu):
            out[np.argmin(np.abs(np.asarray(support_deg) - mm)), j] = 1.0
    else:
        out = np.exp(k * np.cos(x - u) - k) / (2 * np.pi * i0e(k))
    if norm:
        out = out / out.sum(0, keepdims=True)
    return out


def girshick_lookup(motdir, k_llh, mode_prior, k_prior, weight_tail=0.0):
    m = DEG
    mPdfs = vm_pdf(m, motdir, k_llh)
    like  = vm_pdf(DEG, m, k_llh)
    prior = np.tile(vm_pdf(DEG, mode_prior, k_prior), (1, len(m)))
    if weight_tail > 0:
        prior = (1 - weight_tail) * prior + weight_tail * (1 / 360)
    post = like * prior
    post = post / post.sum(0, keepdims=True)
    post = np.round(post * 1e6) / 1e6
    percepts = [np.where(post[:, i] == post[:, i].max())[0] + 1 for i in range(len(m))]
    L = np.zeros((360, len(motdir)))
    for i in range(len(m)):
        ps = percepts[i]; w = 1.0 / len(ps)
        for p in ps:
            L[p - 1, :] += w * mPdfs[i, :]
    return DEG, L / L.sum(0, keepdims=True)


def circ_convolve(P, kernel_col):
    Fp = np.fft.fft(P, axis=0)
    Fk = np.fft.fft(kernel_col)[:, None]
    return np.real(np.fft.ifft(Fp * Fk, axis=0))


def trial_loglike(estimates, dirs, cohs, pstds, params):
    """Vectorized inner loop (a python for-loop over trials was the original version,
    it did not scale past a couple of subjects, this is the fixed version)."""
    k_llh_map, k_pri_map = params['k_llh'], params['k_prior']
    p_rand, k_motor = params['p_rand'], params['k_motor']
    mode = params.get('mode_prior', 225)
    T = len(estimates)
    P = np.zeros((360, T))
    for coh in np.unique(cohs):
        for ps in np.unique(pstds):
            sel = (cohs == coh) & (pstds == ps)
            if not sel.any():
                continue
            dd = np.unique(dirs[sel])
            _, L = girshick_lookup(dd, k_llh_map[coh], mode, k_pri_map[ps])
            col = {int(d): i for i, d in enumerate(dd)}
            idx = np.where(sel)[0]
            d_idx = np.array([col[int(d)] for d in dirs[idx]])
            P[:, idx] = L[:, d_idx]
    P = (1 - p_rand) * P + p_rand * (1 / 360)
    motor = vm_pdf(DEG, 360, k_motor)[:, 0]
    P = circ_convolve(P, motor)
    P = np.clip(P, 1e-320, None)
    P = P / P.sum(0, keepdims=True)
    e_idx = (np.round(estimates).astype(int) % 360); e_idx[e_idx == 0] = 360
    p_obs = P[e_idx - 1, np.arange(T)]
    return -np.log(p_obs).sum(), p_obs


cohs_u, ps_u = [0.06, 0.12, 0.24], [80, 40, 20, 10]

def unpack_switching(x):
    return dict(k_llh=dict(zip(cohs_u, x[:3])),
                k_prior=dict(zip(ps_u, x[3:7])),
                p_rand=x[7], k_motor=x[8], mode_prior=225)


def fit_switching_subject(sub_id, max_trials=2500, seed=0):
    """Fits Kp/Ke per subject. Subsampled to max_trials and single Nelder-Mead run for
    speed -- good enough to check direction of an effect, NOT a substitute for the
    paper's own full-trial, multi-restart fitting procedure."""
    sub = data[data.subject_id == sub_id]
    if len(sub) > max_trials:
        sub = sub.sample(n=max_trials, random_state=seed)
    est, dirs = sub.estimate_deg.values, sub.motion_direction.values.astype(float)
    cohs, pstds = sub.motion_coherence.values, sub.prior_std.values

    def obj(x):
        if (x < 0).any() or x[7] > 1:
            return 1e9
        return trial_loglike(est, dirs, cohs, pstds, unpack_switching(x))[0]

    x0 = np.array([1, 3, 8, 0.5, 1, 2.5, 6, 0.02, 30.0])
    res = minimize(obj, x0, method="Nelder-Mead",
                   options=dict(maxiter=600, xatol=5e-2, fatol=1.0))
    return len(est), bool(res.success), float(res.fun), unpack_switching(res.x)


# fit all 12 subjects (takes a few minutes; run in smaller batches if this times out)
switching_fits = {}
for sid in range(1, 13):
    n, ok, nll, params = fit_switching_subject(sid)
    switching_fits[sid] = dict(n_trials=n, success=ok, nll=nll, **params)
    print(f"subject {sid:>2}: n={n:>5} success={ok} nll={nll:.1f}")


In [ ]:
print(f'{"subject":>7} {"80deg":>8} {"40deg":>8} {"20deg":>8} {"10deg":>8}   pattern')
n_monotonic = 0
for sid in range(1, 13):
    r = switching_fits[sid]
    k_llh, k_prior = r['k_llh'], r['k_prior']
    row = []
    for ps in [80, 40, 20, 10]:
        Kp = k_prior[ps]
        p06 = Kp / (Kp + k_llh[0.06])
        p24 = Kp / (Kp + k_llh[0.24])
        row.append((p06, p24))
    ok = all(p06 > p24 for p06, p24 in row)
    n_monotonic += ok
    vals = '  '.join(f'{p06:.2f}->{p24:.2f}' for p06, p24 in row)
    print(f'{sid:>7} {vals}   {"OK" if ok else "NO"}')

print()
print(f'{n_monotonic}/12 subjects show condition_prior_reliance decreasing from 6% to 24% coherence, for every prior width')


**What we found:** confirmed, cleanly, for all 12 subjects, every prior width. `condition_prior_reliance`
drops from 6% to 24% coherence in all 48 subject-x-prior-width comparisons, exactly Varsha's
prediction. Real individual variation underneath that headline number though: subjects 5, 6, and
11 show near-zero reliance on the *widest* (80 deg, sometimes 40 deg) prior even at the lowest
coherence, quite different from subjects like 2, 10, and 12, who lean on that same wide prior
substantially (0.3-0.45) when coherence is low. The decreasing-with-coherence pattern holds for
everyone, but the starting point differs a lot subject to subject.

**Decided:** Varsha's Kp/Ke-by-coherence prediction is confirmed. Worth a fuller fit later (full
trial set, multiple restarts, matching the paper's own convergence-checking approach) before
treating the exact magnitudes as final, this subsampled single-run version is solid enough to
confirm the direction, not precise enough for publication-grade numbers.


## 6. Hunting for the learning rate empirically, before realizing it's a model parameter

Varsha, in the same meeting, on whether the learning rate itself is a free parameter:

"You don't need to fit the learning rate, it's just the prediction of the model... you fit to the
estimated angles [via MLE]... Now, from the model, you will have a learning rate for your prior."
And separately: "correlation over blocks of the hyperprior learning rate and the estimated error
learning rate, and see if they are related." And on coherence specifically: "probably the
learning rate would be different between the three coherences."

That's two things to check: does a learning rate differ by coherence, and does a model-derived
learning rate correlate with an independent, model-free behavioral one. We tried the first one
three different ways before realizing the more direct path was sitting in the model we'd already
built.

### Attempt 1: fit an exponential decay directly to error, rejected

**Tried:** bin trials by within-block position, fit `abs_error(t) = asymptote + (start -
asymptote) * exp(-t / tau)` per subject and coherence, using `scipy.optimize.curve_fit`.

**Found:** 9 of 36 subject-coherence fits (25%) landed exactly on the upper bound we set for
`tau`, that's non-convergence, not a real answer, the optimizer had nothing in the data to
constrain it. Checking the raw numbers directly: several subject-coherence pairs showed error
getting *worse*, not better, across the block (e.g. subject 3 at 12% coherence: 18.1 deg early ->
20.9 deg late). This matches the team's own earlier finding in the RT-exploratory notebook, that
`abs_error_from_motion` shows no significant transition-specific effect (p=0.5536), the same weak
signal was just resurfacing here in a different analysis.

**Rejected:** too fragile to trust; moved to a simpler, boundary-artifact-free measure instead.


In [ ]:
early = data[data.trials_into_block.between(1, 15)]
late  = data[data.trials_into_block >= 101]

def summarize(df, col, label):
    return df.groupby(['subject_id', 'motion_coherence'])[col].mean().rename(label)

early_err = summarize(early, 'abs_error', 'early_mean_error')
late_err  = summarize(late,  'abs_error', 'late_mean_error')
err_combined = pd.concat([early_err, late_err], axis=1).dropna()
err_combined['decline'] = err_combined['early_mean_error'] - err_combined['late_mean_error']

pivot = err_combined.reset_index().pivot(index='subject_id', columns='motion_coherence', values='decline')
print('Error decline (early minus late), positive = error got smaller over the block')
print(pivot.round(1))
print()
print('mean decline by coherence:', pivot.mean().round(2).to_dict())
print('sem  decline by coherence:', (pivot.std() / np.sqrt(pivot.count())).round(2).to_dict())


**Attempt 2 found:** a real, trustworthy measure this time, but a null result. All three
coherence means (1.77, 2.59, 2.08) sit within one SEM of each other and of zero, no ordering
matches the prediction. The real story is individual, not coherence-driven: subject 5 gets worse
across every block regardless of coherence (-4.5, -5.4, -4.0), subject 10 improves substantially
at every coherence (8.4, 16.6, 8.4). Plain error magnitude doesn't carry the signal we're looking
for.

### Attempt 3: `signed_prior_pull`, the team's own more targeted measure

The team's own RT-exploratory notebook already anticipated attempt 2's failure and named the fix,
whether the error points *toward* the prior specifically, not just its raw size.


In [ ]:
direction_to_prior = np.sign(-data.motion_offset_from_prior)  # sign(prior_mean - motion_direction)
data['signed_prior_pull'] = direction_to_prior * data.error_deg

# exclude trials where prior and motion nearly coincide -- pull is undefined/noisy there
pull_data = data[data.motion_offset_from_prior.abs() >= 10]

early_pull = summarize(pull_data[pull_data.trials_into_block.between(1, 15)], 'signed_prior_pull', 'early_pull')
late_pull  = summarize(pull_data[pull_data.trials_into_block >= 101],        'signed_prior_pull', 'late_pull')
pull_combined = pd.concat([early_pull, late_pull], axis=1).dropna()
pull_combined['pull_increase'] = pull_combined['late_pull'] - pull_combined['early_pull']

pivot2 = pull_combined.reset_index().pivot(index='subject_id', columns='motion_coherence', values='pull_increase')
print('Increase in prior-pull across the block (late minus early), positive = leaning on prior MORE by the end')
print(pivot2.round(1))
print()
print('mean pull_increase by coherence:', pivot2.mean().round(2).to_dict())
print('sem  pull_increase by coherence:', (pivot2.std() / np.sqrt(pivot2.count())).round(2).to_dict())


**Attempt 3 found:** a real, coherence-dependent pattern, finally, but not the one predicted.
6% coherence stays roughly flat (-1.56, within its own SEM of zero). 12% and 24% both show a real
*decrease* in prior-pull across the block (-4.38 and -3.88, both 3-4x their own SEM away from
zero). That's backwards from "confidence builds, so pull should increase", it looks more like
subjects lean on the prior somewhat at the start of every block regardless of that trial's
coherence, maybe a general settling-in effect, and that early pull fades specifically once good
evidence accumulates, while it doesn't fade the same way when evidence stays weak throughout.

**Decided:** stop hunting for the learning rate behaviorally. It's not a mystery quantity that
needs indirect triangulation, it's a literal parameter of the model already built above,
`alpha` inside `update_prior_reliance_windowed`. Once that gets fit per subject via MLE (matching
exactly what Varsha described: fit to the estimated-angle distribution, the learning rate falls
out as a result, not a separate thing to hunt for first), the fitted value *is* the model-derived
learning rate, and correlating it against attempt 3's `pull_increase` becomes a real, well-posed
validation check, not a shot in the dark. That NLL and fitting loop is the next thing to build,
not more behavioral detective work.


## Summary

### Decided
- Block = `(subject_id, run_id)`, stored as `block_id`
- `experiment_id` 11/12 is one combined study per subject, not two separate studies
- Model architecture: discrete mixture (reliance-weighted sum of prior and likelihood
  components), not the continuous-width multiplicative version
- `prior_reliance` is learned via a delta rule driven by per-trial agreement with the prior

### Still open
1. Real inter-block timing -- ask Salma/Varsha whether timestamps exist anywhere outside this CSV
2. Window size for smoothing `agreement_t` (tested 5, not chosen/fitted)
3. **Immediate next step:** build the real NLL for the reliability-mixture model and fit it per
   subject via `scipy.optimize`, this gives `alpha` (the model-derived learning rate) as a fitted
   output, satisfying the actual task and enabling a real correlation against the behavioral
   `pull_increase` measure from Section 6
4. Not yet verified: whether teammates' block-transition-labeling code (`transition_type`,
   `prev_prior_std`) already accounts for the trial-index/trial-time block-reset issue found
   above
5. Once the NLL exists: correlate the fitted learning rate against `pull_increase` per subject
   (Section 6), and re-run the AIC-style comparison against the paper's own Switching observer

### Decided since (session-boundary window reset)
- The window resets only at session boundaries (`same_session_prev == False`), verified against
  `hb_verified_model_implementation.ipynb`'s own `same_session_prev` convention. Within a session,
  it carries across block boundaries on purpose, that's the carryover mechanism being tested, not
  something to suppress by default.

### Decided since (Kp/Ke and the learning rate)
- Varsha's Kp/Ke-by-coherence prediction: confirmed, 12/12 subjects, every prior width.
- The learning rate is not something to reverse-engineer from behavior first, it is `alpha`, a
  parameter of the model already built in Section 4. Fit it directly; three behavioral proxies
  were tried in Section 6 as a detour before landing on this.
